# 🔴 HIV/AIDS Global Analytics – Exploratory Data Analysis
**Course:** Exploratory Data Analysis  
**Instructor:** Ali Hassan Sherazi  
**Dataset:** `aidsinfo.unaids.org.csv` — UNAIDS Global Estimates 2025  
**Author:** Faiqa Eman

---

This notebook performs a complete EDA on the UNAIDS HIV/AIDS dataset including:
1. Dataset loading
2. Shape inspection
3. Column analysis
4. Missing value analysis
5. Duplicate check
6. Datatype inspection
7. Data cleaning
8. Transformations
9. Exploratory Data Analysis (charts)
10. Key insights

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.facecolor'] = '#161B22'
plt.rcParams['figure.facecolor'] = '#0D1117'
plt.rcParams['text.color'] = '#E6EDF3'
plt.rcParams['axes.labelcolor'] = '#E6EDF3'
plt.rcParams['xtick.color'] = '#E6EDF3'
plt.rcParams['ytick.color'] = '#E6EDF3'

PALETTE = ['#C0392B','#2980B9','#16A085','#8E44AD','#E67E22',
           '#27AE60','#2C3E50','#D35400','#1ABC9C','#E74C3C']

print('Libraries loaded successfully.')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Seaborn : {sns.__version__}')

## 2. Dataset Loading

In [ ]:
# Load the dataset
# WHY low_memory=False: prevents DtypeWarning on large mixed-type CSVs
DATA_PATH = '../data/aidsinfo.unaids.org.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Dataset loaded: {DATA_PATH}')
print(f'Rows   : {df_raw.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]}')

In [ ]:
# Preview first 5 rows
df_raw.head()

## 3. Shape Inspection

In [ ]:
print('=== SHAPE INSPECTION ===')
print(f'Total rows    : {df_raw.shape[0]:,}')
print(f'Total columns : {df_raw.shape[1]}')
print(f'Total cells   : {df_raw.shape[0] * df_raw.shape[1]:,}')
print()
print('Column names:')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  {i:2d}. {col}')

## 4. Column Analysis

In [ ]:
print('=== COLUMN ANALYSIS ===')
print()

for col in df_raw.columns:
    n_unique = df_raw[col].nunique()
    dtype = str(df_raw[col].dtype)
    sample = df_raw[col].dropna().iloc[0] if len(df_raw[col].dropna()) > 0 else 'N/A'
    print(f'  [{col}]')
    print(f'    dtype   : {dtype}')
    print(f'    unique  : {n_unique:,}')
    print(f'    sample  : {sample}')
    print()

In [ ]:
# Categorical columns — value distribution
cat_cols = ['Indicator', 'Unit', 'Subgroup', 'Area', 'Source']

for col in cat_cols:
    print(f'=== {col} === (top 10)')
    print(df_raw[col].value_counts().head(10).to_string())
    print()

## 5. Missing Value Analysis

In [ ]:
print('=== MISSING VALUE ANALYSIS ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)
print(missing_df.to_string())

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(10, 5))
missing_pct_nz = missing_pct[missing_pct > 0].sort_values(ascending=True)
bars = ax.barh(missing_pct_nz.index, missing_pct_nz.values, color=PALETTE[0], edgecolor='#0D1117')
ax.set_xlabel('Missing %')
ax.set_title('Missing Value Percentage by Column', color='#C0392B', fontsize=13)
for bar, val in zip(bars, missing_pct_nz.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', color='#E6EDF3', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Duplicate Check

In [ ]:
print('=== DUPLICATE CHECK ===')
n_dups = df_raw.duplicated().sum()
print(f'Total duplicate rows : {n_dups:,}')
print(f'Duplicate percentage : {n_dups/len(df_raw)*100:.4f}%')

if n_dups > 0:
    print('Sample duplicate rows:')
    print(df_raw[df_raw.duplicated(keep=False)].head(4).to_string())

## 7. Datatype Inspection

In [ ]:
print('=== DATATYPE INSPECTION ===')
print(df_raw.dtypes)
print()
print('Summary statistics for numeric columns:')
print(df_raw.describe().to_string())

## 8. Data Cleaning

In [ ]:
# Work on a copy to preserve raw data
df = df_raw.copy()
print(f'Starting rows: {len(df):,}')

# Step 1: Drop low-value columns
# WHY: Indicator_GId / Subgroup_Val_GId are internal IDs.
#      Data_Denominator is 99% NaN. Footnote is 64% NaN.
drop_cols = ['Indicator_GId', 'Subgroup_Val_GId', 'Data_Denominator', 'Footnote']
df.drop(columns=drop_cols, errors='ignore', inplace=True)
print(f'After dropping low-value columns: {df.shape[1]} columns remain')

# Step 2: Rename columns with spaces
# WHY: Enables cleaner column access without bracket notation for spaces.
df.rename(columns={
    'Data value': 'Data_Value',
    'Area ID':    'Area_ID',
    'Area Level': 'Area_Level',
    'Time Period': 'Time_Period',
}, inplace=True)
print('Columns renamed successfully.')

# Step 3: Drop rows with NaN Data_Value
# WHY: 626K rows lack the primary measure — they carry no quantitative info.
before = len(df)
df.dropna(subset=['Data_Value'], inplace=True)
print(f'After dropping NaN Data_Value: {before - len(df):,} rows removed → {len(df):,} remain')

# Step 4: Remove negative Data_Value
# WHY: Negative rates/counts are physical impossibilities — data artefacts.
before = len(df)
df = df[df['Data_Value'] >= 0].copy()
print(f'After removing negative values: {before - len(df):,} rows removed → {len(df):,} remain')

# Step 5: Strip whitespace from string columns
# WHY: Prevents false duplicates in groupby (e.g. 'Kenya ' vs 'Kenya').
str_cols = ['Indicator', 'Unit', 'Subgroup', 'Area', 'Source']
for col in str_cols:
    df[col] = df[col].str.strip().fillna('Unknown')
print('String columns stripped and NaN filled.')

# Step 6: Coerce numeric types
# WHY: Ensures downstream arithmetic operations work without dtype errors.
df['Data_Value']  = pd.to_numeric(df['Data_Value'],  errors='coerce')
df['Time_Period'] = pd.to_numeric(df['Time_Period'], errors='coerce')
df['Area_Level']  = pd.to_numeric(df['Area_Level'],  errors='coerce')
df.dropna(subset=['Data_Value', 'Time_Period'], inplace=True)

# Step 7: Reset index
df.reset_index(drop=True, inplace=True)

print(f'\n✅ Final clean dataset: {len(df):,} rows × {df.shape[1]} columns')

## 9. Transformations

In [ ]:
# Transform 1: Area Level numeric → readable label
level_map = {1: 'Global', 2: 'Regional', 3: 'National'}
df['Area_Level_Label'] = df['Area_Level'].map(level_map).fillna('Unknown')
print('Area_Level_Label created:', df['Area_Level_Label'].value_counts().to_dict())

# Transform 2: Decade column from Time_Period
df['Decade'] = (df['Time_Period'] // 10 * 10).astype(int).astype(str) + 's'
print('Decade column created:', df['Decade'].value_counts().to_dict())

# Transform 3: Log-transform of Data_Value for skewed distributions
# WHY: Data_Value spans 0 to ~45M; log scale makes patterns visible
df['Log_Data_Value'] = np.log1p(df['Data_Value'])  # log1p handles zeros safely
print('Log_Data_Value column created (log1p transform).')

# Transform 4: Source shortened for display
df['Source_Short'] = df['Source'].str[:35]

print(f'\n✅ Transformations complete. Final shape: {df.shape}')
df.head(3)

## 10. Exploratory Data Analysis

### 10.1 Chart 1 — Pie Chart: Distribution of HIV Indicator Units

In [ ]:
counts = df['Unit'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=PALETTE[:len(counts)],
    startangle=140,
    wedgeprops=dict(edgecolor='#0D1117', linewidth=1.5)
)
for t in texts + autotexts:
    t.set_color('#E6EDF3')
ax.set_title('Distribution of HIV Indicator Units', color='#C0392B', fontsize=14)
plt.tight_layout()
plt.show()
print('Insight: Rate dominates the dataset, confirming normalised measurements are preferred for cross-country comparison.')

### 10.2 Chart 2 — Histogram: Data Value Distribution

In [ ]:
cap = df['Data_Value'].quantile(0.99)
sub = df[df['Data_Value'] <= cap]['Data_Value']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
sns.histplot(sub, bins=40, kde=True, color=PALETTE[0], edgecolor='#0D1117', ax=axes[0])
axes[0].set_title('Raw Data Value Distribution (99th pct cap)', color='#C0392B', fontsize=12)
axes[0].set_xlabel('Data Value')

# Log-transformed distribution
sns.histplot(df['Log_Data_Value'], bins=40, kde=True, color=PALETTE[1], edgecolor='#0D1117', ax=axes[1])
axes[1].set_title('Log-Transformed Data Value Distribution', color='#C0392B', fontsize=12)
axes[1].set_xlabel('log1p(Data Value)')

plt.tight_layout()
plt.show()
print('Insight: Raw distribution is severely right-skewed. Log transform reveals a near-normal distribution with two peaks.')

### 10.3 Chart 3 — Line Chart: Global HIV/AIDS Trends Over Time

In [ ]:
yearly = df.groupby('Time_Period')['Data_Value'].agg(['mean', 'median', 'std']).reset_index().sort_values('Time_Period')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly['Time_Period'], yearly['mean'], color='#C0392B', linewidth=2.5, marker='o', markersize=4, label='Mean')
ax.plot(yearly['Time_Period'], yearly['median'], color='#2980B9', linewidth=2, linestyle='--', marker='s', markersize=4, label='Median')
ax.fill_between(yearly['Time_Period'], yearly['mean'] - yearly['std'], yearly['mean'] + yearly['std'],
                alpha=0.1, color='#C0392B', label='±1 Std Dev')
ax.set_xlabel('Year')
ax.set_ylabel('Data Value')
ax.set_title('Global HIV/AIDS Average Data Value Trend (1990–2024)', color='#C0392B', fontsize=13)
ax.legend(facecolor='#161B22', labelcolor='#E6EDF3', edgecolor='#30363D')
plt.tight_layout()
plt.show()
print('Insight: Mean values rose through the 1990s-2000s (epidemic expansion), then began stabilising as ART scaled globally.')

### 10.4 Chart 4 — Bar Chart: Top 10 Areas by Average Data Value

In [ ]:
country_df = df[df['Area_Level'] >= 2]
top_areas = country_df.groupby('Area')['Data_Value'].mean().sort_values(ascending=False).head(10).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_areas['Area'], top_areas['Data_Value'], color=PALETTE[:10], edgecolor='#0D1117')
ax.invert_yaxis()
ax.set_xlabel('Average Data Value')
ax.set_title('Top 10 Reporting Areas by Average Data Value', color='#C0392B', fontsize=13)
for bar, val in zip(bars, top_areas['Data_Value']):
    ax.text(val * 1.01, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', va='center', color='#E6EDF3', fontsize=8)
plt.tight_layout()
plt.show()
print('Insight: Areas with highest average values tend to be high-population countries where absolute counts dominate.')

### 10.5 Chart 5 — Scatter Plot: Data Values Over Time by Unit

In [ ]:
sub = df.sample(n=min(20000, len(df)), random_state=42)
cap = sub['Data_Value'].quantile(0.95)
sub = sub[sub['Data_Value'] <= cap]

fig, ax = plt.subplots(figsize=(12, 5))
for i, unit in enumerate(df['Unit'].unique()[:3]):
    s = sub[sub['Unit'] == unit]
    ax.scatter(s['Time_Period'], s['Data_Value'], label=unit,
               color=PALETTE[i], alpha=0.4, s=12, edgecolors='none')
ax.set_xlabel('Year')
ax.set_ylabel('Data Value')
ax.set_title('HIV/AIDS Data Values Over Time by Unit Type', color='#C0392B', fontsize=13)
ax.legend(facecolor='#161B22', labelcolor='#E6EDF3', edgecolor='#30363D')
plt.tight_layout()
plt.show()
print('Insight: Number values are orders of magnitude larger than Rate/Percent — never mix unit types in a single mean calculation.')

### 10.6 Chart 6 — Box Plot: Data Value Spread by Unit Type

In [ ]:
cap = df['Data_Value'].quantile(0.95)
sub = df[df['Data_Value'] <= cap]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=sub, x='Unit', y='Data_Value', hue='Unit', palette=PALETTE[:3], legend=False,
            ax=ax, flierprops=dict(marker='o', color='#C0392B', markersize=3))
ax.set_xlabel('Unit Type')
ax.set_ylabel('Data Value')
ax.set_title('Data Value Distribution by Unit Type', color='#C0392B', fontsize=13)
plt.tight_layout()
plt.show()
print('Insight: Number type has extreme outliers (global counts reach 45M+); Rate and Percent are compact 0–100 ranges.')

### 10.7 Chart 7 — Heatmap: Correlation Between HIV/AIDS Indicators

In [ ]:
top_indicators = df['Indicator'].value_counts().head(8).index.tolist()
pivot_df = (
    df[df['Indicator'].isin(top_indicators)]
    .groupby(['Indicator', 'Time_Period'])['Data_Value']
    .mean()
    .unstack(level=0)
    .dropna(how='all')
)
corr = pivot_df.corr()
corr.columns = [c[:22] for c in corr.columns]
corr.index   = [i[:22] for i in corr.index]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, linecolor='#0D1117',
            annot_kws={'size': 7}, ax=ax)
ax.set_title('Correlation Between Top HIV/AIDS Indicators Over Time', color='#C0392B', fontsize=13)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout()
plt.show()
print('Insight: AIDS-related deaths and new HIV infections are strongly positively correlated, confirming disease progression link.')

### 10.8 Chart 8 — Area Chart: Cumulative Records Over Time

In [ ]:
yearly_counts = df.groupby('Time_Period').size().reset_index(name='Count').sort_values('Time_Period')
yearly_counts['Cumulative'] = yearly_counts['Count'].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual count
axes[0].bar(yearly_counts['Time_Period'], yearly_counts['Count'], color=PALETTE[0], edgecolor='#0D1117')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Records per Year')
axes[0].set_title('Annual HIV/AIDS Records Reported', color='#C0392B', fontsize=12)

# Cumulative area
axes[1].fill_between(yearly_counts['Time_Period'], yearly_counts['Cumulative'], color=PALETTE[0], alpha=0.7)
axes[1].plot(yearly_counts['Time_Period'], yearly_counts['Cumulative'], color=PALETTE[1], linewidth=2)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Cumulative Records')
axes[1].set_title('Cumulative HIV/AIDS Records Over Time', color='#C0392B', fontsize=12)

plt.tight_layout()
plt.show()
print('Insight: Record volume grew exponentially post-2000, reflecting expanded global AIDS monitoring infrastructure.')

### 10.9 Chart 9 — Count Plot: Source Contribution Analysis

In [ ]:
order = df['Source_Short'].value_counts().index

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, y='Source_Short', hue='Source_Short', order=order, legend=False,
              palette=PALETTE[:len(order)], ax=ax, edgecolor='#0D1117')
ax.set_xlabel('Record Count')
ax.set_ylabel('Data Source')
ax.set_title('Source Contribution Analysis', color='#C0392B', fontsize=13)
plt.tight_layout()
plt.show()
print('Insight: UNAIDS Estimates dominates with >90% of records, indicating centralised epidemiological modelling.')

### 10.10 Chart 10 — Violin Plot: Data Value by Area Level

In [ ]:
cap = df['Data_Value'].quantile(0.90)
sub = df[df['Data_Value'] <= cap].sample(n=min(20000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=sub, x='Area_Level_Label', y='Data_Value',
               hue='Area_Level_Label', order=['Global','Regional','National'],
               palette=PALETTE[:3], inner='quartile', legend=False, ax=ax)
ax.set_xlabel('Area Level')
ax.set_ylabel('Data Value')
ax.set_title('HIV/AIDS Data Value Distribution by Area Level', color='#C0392B', fontsize=13)
plt.tight_layout()
plt.show()
print('Insight: National-level data shows widest spread — individual country burdens differ vastly; global aggregates are narrow.')

### 10.11 BONUS — Pair Plot: Key HIV/AIDS Variables

In [ ]:
cap = df['Data_Value'].quantile(0.95)
sub = df[df['Data_Value'] <= cap].sample(n=3000, random_state=42)
cols = ['Time_Period', 'Data_Value', 'Area_Level', 'Unit']
sub_pp = sub[cols].dropna()

g = sns.pairplot(sub_pp, hue='Unit',
                 palette={u: PALETTE[i] for i, u in enumerate(sub_pp['Unit'].unique()[:3])},
                 plot_kws=dict(alpha=0.4, s=10),
                 diag_kind='kde',
                 vars=['Time_Period', 'Data_Value', 'Area_Level'])
g.figure.suptitle('Pair Plot: HIV/AIDS Key Variables by Unit Type', y=1.02, color='#C0392B', fontsize=13)
plt.show()
print('Insight: Clear scale separation between Number and Rate/Percent units across all pairwise combinations.')

## 11. Key Insights Summary

In [ ]:
print('=== KEY INSIGHTS FROM HIV/AIDS EDA ===')
print()
print(f'1. Dataset Size          : {len(df):,} clean records across {df["Area"].nunique()} areas')
print(f'2. Time Span             : {int(df["Time_Period"].min())} – {int(df["Time_Period"].max())} ({int(df["Time_Period"].max()) - int(df["Time_Period"].min()) + 1} years)')
print(f'3. Indicators Covered    : {df["Indicator"].nunique()} unique HIV/AIDS indicators')
print(f'4. Dominant Unit Type    : {df["Unit"].value_counts().index[0]} ({df["Unit"].value_counts().iloc[0]/len(df)*100:.1f}% of records)')
print(f'5. Primary Data Source   : {df["Source_Short"].value_counts().index[0]}')
print(f'6. Average Data Value    : {df["Data_Value"].mean():,.2f}')
print(f'7. Median Data Value     : {df["Data_Value"].median():,.2f}')
print(f'8. Max Data Value        : {df["Data_Value"].max():,.0f}')
print(f'9. Most Records by Area  : {df["Area"].value_counts().index[0]} ({df["Area"].value_counts().iloc[0]:,} records)')
print(f'10. Area Level Breakdown : {df["Area_Level_Label"].value_counts().to_dict()}')
print()
print('=== ANALYTICAL CONCLUSIONS ===')
print()
insights = [
    '1. DATA GROWTH: HIV/AIDS surveillance records grew exponentially post-2000, reflecting global monitoring expansion.',
    '2. UNIT DOMINANCE: Rate measurements dominate, enabling cross-country normalised comparisons.',
    '3. SKEWED DISTRIBUTION: Data_Value is severely right-skewed; log transformation reveals near-normal shape.',
    '4. REGIONAL DISPARITY: Sub-Saharan African countries show highest absolute HIV/AIDS burden.',
    '5. ART IMPACT: Post-2010 indicators show declining mortality as ART coverage expanded globally.',
    '6. INDICATOR CORRELATION: AIDS deaths and new infections are strongly correlated — disease progression link confirmed.',
    '7. AGGREGATION EFFECT: National-level data has widest variance; global aggregates compress heterogeneity.',
    '8. SOURCE CONCENTRATION: UNAIDS Estimates provides 90%+ of records — centralised modelling dominates the dataset.',
]
for insight in insights:
    print(f'  {insight}')